# CVE Agent: Model Server on Colab

Runs Qwen2.5-3B on a Colab GPU and exposes it through a Cloudflare tunnel.
Your PC's Streamlit app sends it the conversation and gets the model's reply back.

**Before you start:** Runtime > Change runtime type > pick a GPU (L4 or A100). Then run each cell in order.

In [6]:
# Cell 1: confirm we have a GPU
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > pick a GPU, then rerun."
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA A100-SXM4-40GB


In [7]:
# Cell 2: install tools (a minute or so)
!pip install -q -U transformers fastapi "uvicorn[standard]"
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

/usr/local/bin/cloudflared: Text file busy


In [8]:
# Cell 3: load the model onto the GPU
import secrets
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
API_KEY = secrets.token_urlsafe(16)  # a random password, so strangers can't use your GPU

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to("cuda")
model.eval()
print("Model loaded on", model.device)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded on cuda:0


In [9]:
# Cell 4: the web API that your PC will call
import secrets
import threading
import torch
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

class GenerateRequest(BaseModel):
    messages: list[dict]
    tools: list[dict] | None = None
    max_new_tokens: int = 400

app = FastAPI()

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID}

@app.post("/generate")
def generate(req: GenerateRequest, x_api_key: str = Header(default="")):
    if not secrets.compare_digest(x_api_key, API_KEY):
        raise HTTPException(status_code=401, detail="Bad API key")
    inputs = tokenizer.apply_chat_template(
        req.messages,
        tools=req.tools,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=req.max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return {"text": tokenizer.decode(new_tokens, skip_special_tokens=True).strip()}

# Run the server in a background thread so the notebook stays usable
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
print("API server started on port 8000")

API server started on port 8000


ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use


In [10]:
# Cell 5: open the tunnel and print the two values to paste into the Streamlit sidebar
import re
import subprocess

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
public_url = None
for line in tunnel.stdout:
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("=" * 60)
print("Paste these into the Streamlit sidebar:")
print("  Colab URL:", public_url)
print("  API key:  ", API_KEY)
print("=" * 60)
print("Keep this notebook running while you use the app.")

Paste these into the Streamlit sidebar:
  Colab URL: https://statutes-encryption-arabic-lance.trycloudflare.com
  API key:   yasFB9VJE1e-9kggqlC1NQ
Keep this notebook running while you use the app.


**Troubleshooting**
- *URL doesn't work right away:* wait ~10 seconds after it prints, then retry.
- *Colab disconnected:* rerun all cells. You'll get a **new URL and key** each time.
- Stop the tunnel and server with Runtime > Interrupt session / Disconnect.